![image.png](https://i.imgur.com/4fN73lZ.png)

## Setup

We install with **uv** and use `imageio` for GIF rendering.

In [ ]:
# Fast dependency install with uv (https://docs.astral.sh/uv).
# Bootstraps uv via pip, then installs into the current kernel/venv. If you already
# run from the project's .venv (created with `uv sync`), this is essentially a no-op.
import sys, os
%pip install -q uv
_target = "" if (sys.prefix != sys.base_prefix or os.environ.get("VIRTUAL_ENV")) else "--system"
!uv pip install -q {_target} --python "{sys.executable}" gymnasium flappy_bird_gymnasium imageio matplotlib torch

# Content

In this lab we build a **GRPO** (Group Relative Policy Optimization) agent from scratch in PyTorch and train it on the Flappy Bird game. Then, in **Part 2**, we put the critic back to get **PPO** and race the two on the same budget.

For the environment we use `flappy_bird_gymnasium`. You can read more about it [here](https://github.com/markub3327/flappy-bird-gymnasium).

> GRPO is built on the *same* PPO-Clip objective as PPO (Part II of today's lecture), with one change — **the critic is gone**.

## GRPO

> **Note:** This is a complete **GRPO** implementation. It uses the PPO-Clip objective from the lecture; the difference from PPO is that the **critic is removed** and the advantage comes from comparing each episode's return against its **group** (`group_advantages`). The student version with tasks to fill in is `04_GRPO_Flappy_Bird_Exercise.ipynb`, one folder up. Part 2 adds the critic back (PPO) and compares the two.

## GRPO: PPO without a critic

Recall the Actor-Critic loop from the lecture: the actor learns a policy and the **critic** $V_\phi(s)$ learns a value baseline. Training two networks at once is the main source of instability, and the critic is roughly half the memory.

**Group Relative Policy Optimization** (GRPO; Shao et al., 2024) throws the critic away. Instead of asking a value network "how good was this state?", it asks a much simpler question: *how good was this whole run compared to a batch of sibling runs from the same policy?*

Concretely, per "prompt/state" it samples a **group** of $G$ outputs under the frozen policy $\pi_{\theta_\text{old}}$ and scores each with a reward $R_i$. The advantage is just the reward centred on the **group mean** — a Monte-Carlo baseline, no value function:

$$\hat{A}_i = \frac{R_i - \overbrace{\mathrm{mean}(R_1,\dots,R_G)}^{\text{the ``group relative'' baseline}}}{\underbrace{\mathrm{std}(R_1,\dots,R_G)}_{\text{optional per-group normaliser}}}$$

Two things to keep straight (straight from the lecture):

- The **mean-subtraction** *is* GRPO — it is the critic-free baseline replacing $V_\phi$.
- The **std-division** is a *separate, optional* stabiliser (the same per-batch advantage normalisation used in PPO), applied per group. A Monte-Carlo baseline does not *need* it.

That single scalar $\hat{A}_i$ is then shared by **every timestep of output $i$** and plugged into the **exact same PPO-Clip objective** from the lecture.

### Mapping GRPO onto a game

GRPO was designed for LLMs, where a "prompt" yields $G$ sampled completions each earning one scalar reward. We map that onto Flappy Bird the natural way:

| GRPO / RLHF term | Here (Flappy Bird) |
| --- | --- |
| prompt / state | the reset environment |
| one output / completion | one **full episode** |
| reward $R_i$ of the output | the **episode return** (its score) |
| a group of $G$ outputs | $G$ episodes rolled from the same policy |

So one *update* = collect $G$ episodes, score each by its return, center on the group mean, and take PPO-Clip steps. This is the same shape as the **RLHF** pipeline in Part III of the lecture — a reward-model score per completion, group-normalised, fed to PPO-Clip.

> **A caveat worth knowing.** Critic-free GRPO leans on episodes *terminating* to separate good runs from bad ones (Flappy ends the moment the bird crashes, so this holds). But every timestep in an episode shares one advantage, so credit assignment is coarse — expect the bird to learn to flap and clear the first pipe or two, not a high score. A critic-based method such as PPO would learn faster; Part 2 puts that to the test.

### The KL-to-reference term (optional here)

The full GRPO objective adds one more piece: a **KL penalty to a fixed reference policy**, $\beta\,\mathrm{KL}(\pi_\theta \,\|\, \pi_\text{ref})$. In **RLHF** that reference is the pre-RL (SFT) model, and the KL is what stops the policy drifting into gibberish that games the learned reward model.

In a from-scratch control task there is no meaningful "SFT model" to anchor to, and PPO-Clip already keeps each step conservative — so we leave the KL **off by default** (`beta_kl = 0.0`). We still snapshot the initial policy as a reference so you can switch it on:

> **Try it:** set `beta_kl` to a small value (e.g. `0.01`) and re-run. Watch how it slows the policy's drift from where it started. It is not needed to learn Flappy, but wiring it up is exactly the mechanism that matters in RLHF (Part III of the lecture).

In [ ]:
import copy
import gymnasium as gym
import flappy_bird_gymnasium
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical

# These tiny networks are env-step bound, so CPU is actually faster than GPU here.
device = torch.device("cpu")
print("device:", device)

## Let's define our model

The network is a policy and nothing else: there is **no critic head**, because GRPO gets its baseline from the group, not from a value network.

In [ ]:
# Policy network: an MLP body with an actor head ONLY.
# There is no critic head -- that is the whole point of GRPO.
class PolicyNet(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, 32)
        self.actor = nn.Linear(32, output_dim)   # action logits (no value head)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        return self.actor(x)

In [ ]:
class GRPOAgent:
    def __init__(self, input_dim, output_dim, lr=1e-3, epsilon=0.2, k_epochs=8,
                 entropy_coef=0.01, beta_kl=0.0, device="cpu"):
        self.policy = PolicyNet(input_dim, output_dim).to(device)
        self.optimizer = optim.Adam(self.policy.parameters(), lr=lr)
        self.epsilon = epsilon
        self.k_epochs = k_epochs
        self.entropy_coef = entropy_coef
        self.beta_kl = beta_kl            # weight of the optional KL-to-reference term (0 = off)
        self.reference = None             # frozen reference policy, set by snapshot_reference()
        self.device = device

    def snapshot_reference(self):
        """Freeze a copy of the current policy as the KL reference (see the note above)."""
        self.reference = copy.deepcopy(self.policy)
        for p in self.reference.parameters():
            p.requires_grad_(False)

    def select_action(self, state):
        """Sample an action under the current policy; return (action, log_prob).
        The log_prob is kept as the OLD log-prob for the PPO ratio."""
        state = torch.tensor(np.asarray(state), dtype=torch.float32, device=self.device)
        with torch.no_grad():
            dist = Categorical(logits=self.policy(state))
            action = dist.sample()
        return action.item(), dist.log_prob(action)

    def group_advantages(self, episode_returns):
        """The GRPO advantage: one scalar per episode.
        Centre each return on the GROUP MEAN (the critic-free baseline) and
        divide by the group std (the optional per-group normaliser)."""
        R = torch.tensor(episode_returns, dtype=torch.float32, device=self.device)
        return (R - R.mean()) / (R.std() + 1e-8)

    def grpo_loss(self, states, actions, advantages, old_log_probs):
        """The PPO-Clip loss -- with NO value term (there is no critic).
        Optionally adds a KL penalty to the frozen reference policy."""
        dist = Categorical(logits=self.policy(states))
        log_probs = dist.log_prob(actions)

        ratio = torch.exp(log_probs - old_log_probs)             # pi_new / pi_old
        surr1 = ratio * advantages
        surr2 = torch.clamp(ratio, 1 - self.epsilon, 1 + self.epsilon) * advantages
        policy_loss = -torch.min(surr1, surr2).mean()

        entropy = dist.entropy().mean()
        loss = policy_loss - self.entropy_coef * entropy

        if self.beta_kl > 0 and self.reference is not None:      # optional; off by default
            with torch.no_grad():
                ref_log_probs = Categorical(logits=self.reference(states)).log_prob(actions)
            kl = (log_probs - ref_log_probs).mean()
            loss = loss + self.beta_kl * kl

        return loss

    def update(self, states, actions, advantages, old_log_probs):
        """Reuse the group for k_epochs of gradient steps on the GRPO loss."""
        states = torch.tensor(np.array(states), dtype=torch.float32, device=self.device)
        actions = torch.tensor(actions, dtype=torch.int64, device=self.device)
        old_log_probs = torch.stack(old_log_probs).detach().to(self.device)
        advantages = advantages.detach().to(self.device)
        for _ in range(self.k_epochs):
            loss = self.grpo_loss(states, actions, advantages, old_log_probs)
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

## Collecting a group

One *update* consumes a **group** of $G$ episodes rolled out under the current (frozen) policy. We flatten every transition into one big batch, and remember each episode's **return** (its score) and **length** so we can broadcast that episode's single advantage back across its own timesteps.

In [ ]:
def collect_group(env, agent, group_size, max_steps):
    """Roll out `group_size` episodes under the current policy.
    Returns the flattened transitions plus the per-episode returns GRPO compares."""
    states, actions, old_log_probs = [], [], []
    episode_returns, episode_lengths = [], []
    for _ in range(group_size):
        state = env.reset()[0]
        ep_return, ep_len = 0.0, 0
        for _ in range(max_steps):
            action, log_prob = agent.select_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            states.append(state)
            actions.append(action)
            old_log_probs.append(log_prob)
            ep_return += reward
            ep_len += 1
            state = next_state
            if terminated or truncated:
                break
        episode_returns.append(ep_return)
        episode_lengths.append(ep_len)
    return states, actions, old_log_probs, episode_returns, episode_lengths

## Initialize the environment and model

In [ ]:
env = gym.make("FlappyBird-v0", render_mode="rgb_array", use_lidar=False)

input_dim = env.observation_space.shape[0]
output_dim = env.action_space.n

group_size = 8        # G: episodes per update, the group each advantage is computed against
num_updates = 1500    # one update == one group of G episodes
max_steps = 1000

lr = 1e-3
epsilon = 0.2
k_epochs = 8
entropy_coef = 0.01
beta_kl = 0.0         # optional KL-to-reference weight; 0.0 = off (see the note above)

agent = GRPOAgent(input_dim, output_dim, lr, epsilon, k_epochs,
                  entropy_coef, beta_kl, device=device)

## Training the model

Each update: collect a group, turn the $G$ returns into $G$ advantages, **broadcast** each episode's advantage across its own timesteps, and take PPO-Clip steps.

In [ ]:
scores = []
grpo_frames, frames_played = [], 0   # frames played so far: Part 2 gives PPO the same budget
agent.snapshot_reference()   # freeze the KL reference (inert while beta_kl == 0)

for update in range(num_updates):
    states, actions, old_log_probs, episode_returns, episode_lengths = \
        collect_group(env, agent, group_size, max_steps)
    frames_played += sum(episode_lengths)

    # one scalar advantage per episode ...
    advantages = agent.group_advantages(episode_returns)
    # ... broadcast across that episode's own timesteps, aligning with the flat batch
    adv_per_step = torch.cat([advantages[i].repeat(n) for i, n in enumerate(episode_lengths)])

    agent.update(states, actions, adv_per_step, old_log_probs)

    scores.append(np.mean(episode_returns))
    grpo_frames.append(frames_played)
    if (update + 1) % 25 == 0:
        print(f"Update {update + 1:5d} | avg return over group (last 25 updates): "
              f"{np.mean(scores[-25:]):7.2f}")

env.close()

## Reward curve

In [ ]:
window = 25
smoothed = np.convolve(scores, np.ones(window) / window, mode="valid")
plt.figure(figsize=(9, 4))
plt.plot(smoothed)
plt.xlabel("update")
plt.ylabel(f"group-mean return ({window}-update moving avg)")
plt.title("GRPO on Flappy Bird")
plt.grid(alpha=0.3)
plt.show()

## Visualizing model's performance

In [ ]:
# Roll out the trained policy and save it as a GIF (Gymnasium-native)
import os
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")  # headless rendering (e.g. Colab)
import imageio.v2 as imageio
from IPython.display import Image, display

os.makedirs("video", exist_ok=True)
eval_env = gym.make("FlappyBird-v0", render_mode="rgb_array", use_lidar=False)
state = eval_env.reset()[0]
frames = []
for _ in range(2000):
    frames.append(eval_env.render())
    action, _ = agent.select_action(state)
    state, reward, terminated, truncated, _ = eval_env.step(action)
    if terminated or truncated:
        break
eval_env.close()
imageio.mimsave("video/flappy_grpo.gif", frames, fps=24, loop=0)

In [ ]:
display(Image(filename="video/flappy_grpo.gif"))

# Part 2: PPO, the Critic Put Back

The caveat above predicted that a critic-based method would learn faster. Let's test it on the **same budget**: the same game, the same `PolicyNet`, the same clipped loss from TASKS 2 and 3, and exactly as many frames as GRPO just played. Only one thing changes: where the advantage comes from.

The lecture's slide *Where Does $\hat{A}_t$ Come From? Three Answers* lists the options. GRPO used the third, the group baseline. PPO uses the second: a **critic** $V_\phi(s)$, a second network trained to predict the return from any state. One real reward plus the critic's guess for the rest gives **every frame its own advantage**, the TD error:

$$\delta_t = r_t + \gamma\, V_\phi(s_{t+1}) - V_\phi(s_t)$$

That goes straight at GRPO's weak spots on this game:

| | GRPO (Part 1) | PPO (Part 2) |
| --- | --- | --- |
| Advantage | one number for the whole episode | one number per frame |
| Baseline | the group's mean, over episodes on different pipe courses | $V_\phi(s_t)$: what *this* state is worth |
| What an update needs | whole episodes (capped at `max_steps`) | any number of frames: $V_\phi$ estimates the rest |

**GAE.** A single $\delta_t$ leans hard on the critic's guess. *Generalized Advantage Estimation* (Schulman et al., 2016) blends in the later TD errors, each weighted down by $\gamma\lambda$:

$$\hat{A}_t = \delta_t + \gamma\lambda\,\hat{A}_{t+1}$$

With $\lambda = 0$ this is the slide's one-step answer; with $\lambda = 1$ it is the slide's first answer, the sampled return minus a baseline (here $V_\phi$). PPO usually takes $\lambda = 0.95$. When an episode ends at step $t$ there is no next state, so $V_\phi(s_{t+1})$ and $\hat{A}_{t+1}$ are dropped (multiplied by $1 - \text{done}_t$).

The critic itself learns by regression: its target for each frame is $\hat{A}_t + V_\phi(s_t)$, and its mean squared error is added to the PPO-Clip loss.

## The critic and the PPO agent

The critic is a separate small network; the actor stays the exact `PolicyNet` from Part 1. `PPOAgent` inherits from `GRPOAgent`, so `select_action` and `grpo_loss` are reused unchanged. It adds the critic, the per-frame advantages (`advantages`), and an `update` that adds the critic's loss and works through the batch in minibatches.

In [ ]:
# The critic V(s): a separate small MLP that predicts the return from a state.
class ValueNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(input_dim, 128), nn.Tanh(),
                                 nn.Linear(128, 128), nn.Tanh(),
                                 nn.Linear(128, 1))

    def forward(self, x):
        return self.net(x).squeeze(-1)


class PPOAgent(GRPOAgent):
    """GRPO's agent plus a critic. select_action and grpo_loss are inherited unchanged."""

    def __init__(self, input_dim, output_dim, lr=3e-4, epsilon=0.2, k_epochs=10, entropy_coef=0.01,
                 gamma=0.99, lam=0.95, value_coef=0.5, minibatch_size=256, device="cpu"):
        super().__init__(input_dim, output_dim, lr, epsilon, k_epochs, entropy_coef, device=device)
        self.critic = ValueNet(input_dim).to(device)
        self.params = list(self.policy.parameters()) + list(self.critic.parameters())
        self.optimizer = optim.Adam(self.params, lr=lr)   # one optimizer for both networks
        self.gamma, self.lam = gamma, lam
        self.value_coef, self.minibatch_size = value_coef, minibatch_size

    def advantages(self, rewards, values, dones, last_value):
        """Per-frame advantages (GAE) and the critic's regression targets.
        rewards, values, dones: arrays over the rollout; last_value: V of the state after it."""
        T = len(rewards)
        adv = np.zeros(T, dtype=np.float32)
        gae = 0.0
        for t in reversed(range(T)):
            next_value = last_value if t == T - 1 else values[t + 1]
            not_done = 1.0 - dones[t]
            delta = rewards[t] + self.gamma * next_value * not_done - values[t]   # the TD error
            gae = delta + self.gamma * self.lam * not_done * gae      # GAE: blend in the later deltas
            adv[t] = gae
        advantages = torch.as_tensor(adv, device=self.device)
        returns = advantages + torch.as_tensor(values, device=self.device)   # what V should predict
        return advantages, returns

    def update(self, states, actions, old_log_probs, advantages, returns):
        """k_epochs over the rollout in minibatches: the PPO-Clip loss plus the critic's loss."""
        for _ in range(self.k_epochs):
            for idx in torch.randperm(len(states), device=self.device).split(self.minibatch_size):
                adv = advantages[idx]
                adv = (adv - adv.mean()) / (adv.std() + 1e-8)      # per-batch normaliser, as in Part 1
                policy_loss = self.grpo_loss(states[idx], actions[idx], adv, old_log_probs[idx])
                value_loss = F.mse_loss(self.critic(states[idx]), returns[idx])
                loss = policy_loss + self.value_coef * value_loss
                self.optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.params, 0.5)       # keep any one step small
                self.optimizer.step()

## Collecting a rollout

GRPO had to play whole episodes before it could score them. PPO doesn't: it plays a fixed number of frames (`n_steps`), starts a new game whenever the bird crashes, and lets the critic estimate whatever would have come after the last frame. We also keep each finished episode's return, to plot against GRPO.

In [ ]:
def collect_rollout(env, agent, state, ep_return, n_steps):
    """Play n_steps frames under the current policy, starting a new game whenever the bird crashes.
    Returns the batch, the state and running return to continue from, and the returns of the
    episodes that finished during the rollout."""
    states, actions, old_log_probs, rewards, dones, finished = [], [], [], [], [], []
    for _ in range(n_steps):
        action, log_prob = agent.select_action(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        states.append(state)
        actions.append(action)
        old_log_probs.append(log_prob)
        rewards.append(reward)
        dones.append(float(terminated or truncated))
        ep_return += reward
        if terminated or truncated:
            finished.append(ep_return)
            next_state, ep_return = env.reset()[0], 0.0
        state = next_state
    batch = (torch.tensor(np.array(states), dtype=torch.float32, device=agent.device),
             torch.tensor(actions, dtype=torch.int64, device=agent.device),
             torch.stack(old_log_probs),
             np.array(rewards, dtype=np.float32),
             np.array(dones, dtype=np.float32))
    return batch, state, ep_return, finished

## Train PPO on GRPO's budget

`budget` is the number of frames GRPO played in Part 1, so the race is fair. This takes a little longer than GRPO's training, because every update now trains the critic too.

In [ ]:
ppo_env = gym.make("FlappyBird-v0", use_lidar=False)
ppo_agent = PPOAgent(input_dim, output_dim, device=device)
n_steps = 2048                  # frames per update
budget = grpo_frames[-1]        # as many frames as GRPO played in Part 1

state, ep_return = ppo_env.reset()[0], 0.0
ppo_returns, ppo_frames, frames_played, update = [], [], 0, 0
while frames_played < budget:
    (states, actions, old_log_probs, rewards, dones), state, ep_return, finished = \
        collect_rollout(ppo_env, ppo_agent, state, ep_return, n_steps)
    frames_played += n_steps

    with torch.no_grad():   # the critic scores the whole rollout in one pass
        values = ppo_agent.critic(states).cpu().numpy()
        last_value = ppo_agent.critic(torch.tensor(np.asarray(state), dtype=torch.float32, device=device)).item()
    advantages, returns = ppo_agent.advantages(rewards, values, dones, last_value)
    ppo_agent.update(states, actions, old_log_probs, advantages, returns)

    ppo_returns += finished
    ppo_frames += [frames_played] * len(finished)
    update += 1
    if update % 50 == 0 and ppo_returns:
        print(f"Update {update:4d} | {frames_played:9,d} of {budget:,d} frames | "
              f"avg return (last 25 episodes): {np.mean(ppo_returns[-25:]):8.2f}")

ppo_env.close()

## GRPO vs PPO

In [ ]:
def moving_average(x, w=25):
    return np.convolve(x, np.ones(w) / w, mode="valid")

plt.figure(figsize=(9, 4))
plt.plot(grpo_frames[24:], moving_average(scores), label="GRPO (Part 1): group-mean return")
plt.plot(ppo_frames[24:], moving_average(ppo_returns), label="PPO (Part 2): return per episode")
plt.yscale("symlog", linthresh=1)
plt.xlabel("frames played")
plt.ylabel("return (25-point moving average)")
plt.title("Same game, same policy network, same number of frames")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
def evaluate(policy, n_games=20, max_frames=20_000):
    """Greedy play (always the most likely action) on n_games fixed courses.
    Returns the pipes cleared in each game, and how many games were still going at max_frames."""
    eval_env = gym.make("FlappyBird-v0", use_lidar=False)
    pipes, still_going = [], 0
    for k in range(n_games):
        state = eval_env.reset(seed=1_000 + k)[0]
        score, ended = 0, False
        for _ in range(max_frames):
            with torch.no_grad():
                logits = policy(torch.tensor(np.asarray(state), dtype=torch.float32, device=device))
            state, _, terminated, truncated, info = eval_env.step(logits.argmax().item())
            score = info.get("score", score)
            if terminated or truncated:
                ended = True
                break
        pipes.append(score)
        still_going += not ended
    eval_env.close()
    return np.array(pipes), still_going

for name, trained in [("GRPO", agent), ("PPO", ppo_agent)]:
    pipes, still_going = evaluate(trained.policy)
    print(f"{name:4s}: {pipes.mean():6.1f} pipes per game on average, best {pipes.max()} "
          f"({len(pipes)} greedy games; {still_going} still going when stopped at 20,000 frames)")

## Watch PPO play

In [ ]:
# 30 seconds of greedy play; every other frame at half size keeps the GIF small
eval_env = gym.make("FlappyBird-v0", render_mode="rgb_array", use_lidar=False)
state = eval_env.reset(seed=0)[0]
gif_frames = []
for t in range(900):
    if t % 2 == 0:
        gif_frames.append(eval_env.render()[::2, ::2])
    with torch.no_grad():
        logits = ppo_agent.policy(torch.tensor(np.asarray(state), dtype=torch.float32, device=device))
    state, reward, terminated, truncated, _ = eval_env.step(logits.argmax().item())
    if terminated or truncated:
        gif_frames.append(eval_env.render()[::2, ::2])
        break
eval_env.close()
imageio.mimsave("video/flappy_ppo.gif", gif_frames, fps=15, loop=0)

In [ ]:
display(Image(filename="video/flappy_ppo.gif"))

## Takeaways

- **Same game, same policy network, same clipped loss, same number of frames.** The only change was the critic, and it is the difference between clearing a pipe or two and clearing dozens, often hundreds.
- **Why the critic wins here:** every frame gets its own advantage, so the blame for a crash lands on the flaps just before it; the baseline knows what each state is worth instead of comparing different pipe courses; and an update no longer waits for whole episodes.
- **So why does GRPO exist?** It was designed for LLMs, where the trade-off flips: a critic would be another copy of a huge model, the reward arrives once per answer, the group is $G$ answers to the *same* prompt, and the policy starts pretrained and mostly right. Lab 5 uses GRPO in exactly that setting.
- **Try it:** create the agent with `PPOAgent(input_dim, output_dim, lam=0.0, device=device)` (the slide's one-step TD error) or `lam=1.0` (the sampled return minus $V_\phi$) and re-run Part 2. Which learns faster, and why?